# Task 8 — Discrete adjoint optimization for inverse heat-source reconstruction

## Research question
Can we compute gradients of a PDE-constrained objective without building the observation-response matrix one column at a time?

For the explicit finite-difference heat solver, let the interior temperature vector satisfy

$$\mathbf u^{n+1}=\mathbf M\mathbf u^n+\Delta t\,\mathbf B\boldsymbol\theta,\qquad \mathbf u^0=0.$$

Here $\mathbf M=\mathbf I+\alpha\Delta t\,\mathbf L_h$ and $\mathbf B$ evaluates the Gaussian basis functions at interior grid nodes. The observations at selected time levels satisfy $\mathbf y_n=\mathbf H_n\mathbf u^n$. We minimize

$$J(\boldsymbol\theta)=\frac{1}{2m}\sum_n\|\mathbf H_n\mathbf u^n-\mathbf d_n\|_2^2+\frac\lambda2\|\boldsymbol\theta\|_2^2.$$

The **discrete adjoint** uses the recurrence

$$\mathbf p^N=\frac1m\mathbf H_N^\mathsf T(\mathbf H_N\mathbf u^N-\mathbf d_N),$$
$$\mathbf p^n=\mathbf M^\mathsf T\mathbf p^{n+1}+\frac1m\mathbf H_n^\mathsf T(\mathbf H_n\mathbf u^n-\mathbf d_n).$$

When no observations occur at $n$, the second term is zero. The gradient is

$$\boxed{\nabla J(\boldsymbol\theta)=\Delta t\,\mathbf B^\mathsf T\sum_{n=1}^N\mathbf p^n+\lambda\boldsymbol\theta.}$$

This is the adjoint of our **discretized** PDE scheme, not a separately discretized continuous adjoint. The homogeneous Dirichlet Laplacian makes $\mathbf M$ symmetric in this implementation.

The experiments below verify the gradient numerically, reconstruct sources with 25 and 100 coefficients, and compare the work required for one adjoint gradient to building an explicit observation-response matrix. Timing values are machine-dependent; these are different operations, so do not interpret their ratio as a general end-to-end optimizer speedup.

## 1. Grid, PDE solver, observations and adjoint

In [1]:
import numpy as np, time, json
from scipy.optimize import minimize
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
P=Path('.')
alpha=.1; T=.1; nx=24; h=1/nx; nt=64; dt=T/nt
assert 4*alpha*dt/h**2 <=1
xx=np.linspace(0,1,nx+1)[1:-1]; X,Y=np.meshgrid(xx,xx,indexing='ij'); ncell=X.size
obs_nodes=[(i,j) for i in [4,9,14,19] for j in [4,9,14,19]]
obs_times=[16,32,64]; m=len(obs_nodes)*len(obs_times)

def basis(p):
    k=int(np.sqrt(p)); assert k*k==p
    centers=np.linspace(.12,.88,k); sigma=.11 if k==5 else (.07 if k==10 else .045)
    B=np.stack([np.exp(-((X-a)**2+(Y-b)**2)/(2*sigma**2)) for a in centers for b in centers],axis=-1)
    return B.reshape(ncell,p),sigma

def lap(u):
    z=np.pad(u,1)
    return (z[2:,1:-1]+z[:-2,1:-1]+z[1:-1,2:]+z[1:-1,:-2]-4*u)/h**2

def step(u): return u+alpha*dt*lap(u)

def forward(theta,B):
    src=(B@theta).reshape(nx-1,nx-1)
    u=np.zeros_like(src); observations=[]
    for n in range(1,nt+1):
        u=step(u)+dt*src
        if n in obs_times: observations.extend(u[i-1,j-1] for i,j in obs_nodes)
    return np.asarray(observations),u

def adjoint_gradient(theta,B,data,reg):
    y,_=forward(theta,B)
    residual=(y-data)/m
    injections={}
    pos=0
    for n in obs_times:
        a=np.zeros((nx-1,nx-1))
        for i,j in obs_nodes:
            a[i-1,j-1]+=residual[pos];pos+=1
        injections[n]=a
    q=np.zeros((nx-1,nx-1)); cumul=np.zeros_like(q)
    for n in range(nt,0,-1):
        if n in injections: q+=injections[n]
        cumul+=q
        q=step(q) # M symmetric for zero Dirichlet boundaries
    grad=dt*(B.T@cumul.ravel())+reg*theta
    fun=.5*np.dot(y-data,y-data)/m+.5*reg*np.dot(theta,theta)
    return fun,grad



## 2. Central-difference gradient verification

In [2]:
B25,sigma=basis(25); rng=np.random.default_rng(1729)
true25=rng.uniform(.3,1.4,25); clean,_=forward(true25,B25)
noise_sd=.01*np.max(np.abs(clean)); noisy=clean+rng.normal(0,noise_sd,m)
reg=1e-7
start=rng.normal(0,.1,25)
val,grad=adjoint_gradient(start,B25,noisy,reg)
fd=np.zeros(25)
for k in range(25):
    e=np.zeros(25);e[k]=1e-5
    fd[k]=(adjoint_gradient(start+e,B25,noisy,reg)[0]-adjoint_gradient(start-e,B25,noisy,reg)[0])/(2e-5)
relgrad=np.linalg.norm(grad-fd)/max(np.linalg.norm(fd),1e-14)
print('fd relerror',relgrad,'max',np.max(abs(grad-fd)))



fd relerror 5.5157569722909345e-11 max 6.011597469823826e-14


## 3. Inverse optimization and response-matrix comparison

In [3]:
def opt(p, maxiter=160):
    B,_=basis(p)
    theta_true=np.random.default_rng(p).uniform(.3,1.3,p)
    clean,_=forward(theta_true,B)
    d=clean+np.random.default_rng(p+100).normal(0,.01*np.max(abs(clean)),m)
    counter=[0]; t0=time.perf_counter()
    def fg(t): counter[0]+=1;return adjoint_gradient(t,B,d,reg)
    result=minimize(fg,np.zeros(p),jac=True,method='L-BFGS-B',options={'maxiter':maxiter,'ftol':1e-16,'gtol':1e-13})
    runtime=time.perf_counter()-t0
    return result,B,theta_true,d,runtime,counter[0]
res25,B25,theta25,d25,t25,eval25=opt(25,600)
print('opt25',res25.success,res25.nit,res25.fun,t25,eval25)
# Build explicit response matrix for independent validation and operation comparison.
t0=time.perf_counter()
A25=np.column_stack([forward(np.eye(25)[k],B25)[0] for k in range(25)])
buildtime25=time.perf_counter()-t0
ref25=np.linalg.solve(A25.T@A25/m+reg*np.eye(25),A25.T@d25/m)
optagreement=np.linalg.norm(res25.x-ref25)/max(np.linalg.norm(ref25),1e-15)
print('opt vs direct',optagreement)
# Compare gradient from response matrix and discrete adjoint.
probe=np.linspace(.1,.8,25); ya=A25@probe
Gmat=A25.T@(ya-d25)/m+reg*probe
_,Gadj=adjoint_gradient(probe,B25,d25,reg)
print('matrix gradient relative agreement',np.linalg.norm(Gmat-Gadj)/np.linalg.norm(Gmat))
# larger p=100
res100,B100,theta100,d100,t100,eval100=opt(100,600)
print('opt100',res100.success,res100.nit,res100.fun,t100,eval100)
# single adjoint gradient vs building explicit A for p=100
start=time.perf_counter(); fg100=adjoint_gradient(np.zeros(100),B100,d100,reg); gradtime100=time.perf_counter()-start
start=time.perf_counter(); A100=np.column_stack([forward(np.eye(100)[k],B100)[0] for k in range(100)]); buildtime100=time.perf_counter()-start
matrixg100=A100.T@(-d100)/m
print('cost',gradtime100,buildtime100)


opt25 True 179 2.339057054881235e-06 4.927752847000022 192
opt vs direct 5.0905687375045536e-05
matrix gradient relative agreement 7.980403390968135e-16
opt100 True 211 5.238637380167158e-06 8.480255026999998 224
cost 0.03114522499998884 0.8758013480000386


## 4. Source reconstructions, plots and saved results

In [4]:
# plot
fig,axs=plt.subplots(2,2,figsize=(12,9))
axs[0,0].imshow((B25@theta25).reshape(nx-1,nx-1),origin='lower',extent=[h,1-h,h,1-h]);axs[0,0].set_title('Reference heat source (25 basis functions)')
axs[0,1].imshow((B25@res25.x).reshape(nx-1,nx-1),origin='lower',extent=[h,1-h,h,1-h]);axs[0,1].set_title('Adjoint-optimized reconstructed source')
axs[1,0].plot(grad,label='Discrete adjoint');axs[1,0].plot(fd,'x',label='Central finite difference',markersize=4);axs[1,0].set_title('Gradient consistency check');axs[1,0].set_xlabel('Coefficient index');axs[1,0].legend()
axs[1,1].bar(['Adjoint gradient (100)','Build matrix (100)'],[gradtime100,buildtime100]);axs[1,1].set_ylabel('Elapsed time (s)');axs[1,1].set_title('Cost comparison (different outputs)')
fig.tight_layout();fig.savefig(P/'phase8_adjoint_results.png',dpi=170);plt.close(fig)
metrics=dict(nx=nx,nt=nt,dt=dt,stability=4*alpha*dt/h**2,n_observations=m,finite_difference_gradient_relative_error=relgrad,adjoint_vs_matrix_gradient_relative_error=float(np.linalg.norm(Gmat-Gadj)/np.linalg.norm(Gmat)),optimizer_vs_direct_relative_error=float(optagreement),p25_iterations=int(res25.nit),p25_success=bool(res25.success),p25_objective=float(res25.fun),p25_time_s=t25,p25_gradient_evaluations=eval25,p25_source_relative_error=float(np.linalg.norm(B25@(res25.x-theta25))/np.linalg.norm(B25@theta25)),p100_iterations=int(res100.nit),p100_success=bool(res100.success),p100_time_s=t100,p100_objective=float(res100.fun),p100_gradient_evaluations=eval100,adjoint_gradient_time100_s=gradtime100,explicit_matrix_build_time100_s=buildtime100)
(P/'phase8_metrics.json').write_text(json.dumps(metrics,indent=2))
np.savez_compressed(P/'phase8_results.npz',A25=A25,B25=B25,theta_true25=theta25,theta_est25=res25.x,d25=d25,B100=B100,theta_true100=theta100,theta_est100=res100.x,metrics=np.array(list(metrics.items()),dtype=object))
print(json.dumps(metrics,indent=2))


{
  "nx": 24,
  "nt": 64,
  "dt": 0.0015625,
  "stability": 0.3600000000000001,
  "n_observations": 48,
  "finite_difference_gradient_relative_error": 5.5157569722909345e-11,
  "adjoint_vs_matrix_gradient_relative_error": 7.980403390968135e-16,
  "optimizer_vs_direct_relative_error": 5.0905687375045536e-05,
  "p25_iterations": 179,
  "p25_success": true,
  "p25_objective": 2.339057054881235e-06,
  "p25_time_s": 4.927752847000022,
  "p25_gradient_evaluations": 192,
  "p25_source_relative_error": 0.09430198232673365,
  "p100_iterations": 211,
  "p100_success": true,
  "p100_time_s": 8.480255026999998,
  "p100_objective": 5.238637380167158e-06,
  "p100_gradient_evaluations": 224,
  "adjoint_gradient_time100_s": 0.03114522499998884,
  "explicit_matrix_build_time100_s": 0.8758013480000386
}
